In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
import time
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"학습에 사용할 장치: {device}")

#hyperparameter
batch_size = 64
learning_rate = 0.001
num_epochs = 20

#data transform
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
valid_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

#data load
print("CIFAR-10 데이터셋 다운로드 중...")
train_dataset = torchvision.datasets.CIFAR10(root='./data',
                                             train=True,
                                             download=True,
                                             transform=train_transform)
valid_dataset = torchvision.datasets.CIFAR10(root='./data',
                                             train=False,
                                             download=True,
                                             transform=valid_transform)
train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=2)
valid_loader = DataLoader(valid_dataset,
                          batch_size=batch_size,
                          shuffle=False,
                          num_workers=2)
print("데이터 로더 준비 완료.")


학습에 사용할 장치: cuda
CIFAR-10 데이터셋 다운로드 중...


100%|██████████| 170M/170M [00:04<00:00, 37.2MB/s]


데이터 로더 준비 완료.


In [3]:

#model
model = models.vgg16(pretrained=True)

#print(model)

# 입력 이미지가 32x3x로 작으므로 풀링 제거 튜닝 
model.features[16] = nn.Identity()
model.features[23] = nn.Identity()
model.features[30] = nn.Identity()

in_features = model.classifier[0].in_features # 25088
hidden_dim = 1024 # 4096 대신 1024 사용
model.classifier = nn.Sequential(
    nn.Linear(in_features, hidden_dim), # (0) 25088 -> 1024
    nn.ReLU(inplace=True),              # (1)
    nn.Dropout(p=0.5),                  # (2)
    nn.Linear(hidden_dim, hidden_dim),  # (3) 1024 -> 1024
    nn.ReLU(inplace=True),              # (4)
    nn.Dropout(p=0.5),                  # (5)
    nn.Linear(hidden_dim, 10)           # (6) 1024 -> 10 (CIFAR-10)
)

model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 180MB/s]


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Identity()
    (17): Conv2d(256, 512, kernel_size=(3, 3)

In [8]:
#손실 함수, 옵티마이저, 스케줄러
criterion = nn.CrossEntropyLoss()


# 1. 'features' 블록의 모든 파라미터를 동결(학습 방지)
print("사전 학습된 'features' 블록을 동결(freeze)합니다.")
for param in model.features.parameters():
    param.requires_grad = False

# 2. 'classifier'의 파라미터만 학습하도록 옵티마이저 설정
print("'classifier' 블록의 파라미터만 학습을 위해 옵티마이저에 전달합니다.")
# 옵티마이저가 model.parameters()가 아닌 model.classifier.parameters()를 받도록 수정
optimizer = optim.Adam(model.classifier.parameters(), lr=learning_rate)

### 왜 이렇게 했냐면, 우리가 classifier를 1024로 바꿈에 따라 새로 학습을 진행하면서 classifer쪽 웨이트들이 이미지넷 가지고 pretrained됐던게 없어지며 완전히 새로 만들어지고
### 백 프로파게이션하면서 features의 웨이트까지 완전히 망가지게 됨. 
### 따라서 이미지넷으로 학습한 결과 features 웨이트는 고정시키고(단, 풀링만 없애고)
### classifier만 cifar-10에 맞게 조정하고 소형화해서 재 학습 시키면서 파인튜닝함
### 결과적으로, 총 웨이트 파일이 우리 보드 DRAM제한인 512MB를 넘지 않으며 cifar-10 분류를 할 수 있게함.

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=6, gamma=0.2)

#학습
print(f"--- {num_epochs} Epoch Training Start (Classifier만 학습) ---")

total_start_time = time.time()
best_accuracy = 0.0

for epoch in range(num_epochs):
    epoch_start_time = time.time()

    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_train_acc = 100 * correct_train / total_train

    #검증
    model.eval()
    correct_valid = 0
    total_valid = 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_valid += labels.size(0)
            correct_valid += (predicted == labels).sum().item()

    epoch_valid_acc = 100 * correct_valid / total_valid
    scheduler.step()
    epoch_end_time = time.time()
    epoch_duration = epoch_end_time - epoch_start_time

    print(f"--- Epoch {epoch+1}/{num_epochs} 완료 (소요 시간: {epoch_duration:.2f}초) ---")
    print(f"Training Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_train_acc:.2f} %")
    print(f"★ Validation Accuracy: {epoch_valid_acc:.2f}% ★")


    if epoch_valid_acc > best_accuracy:
        best_accuracy = epoch_valid_acc

        torch.save(model.state_dict(), '/content/drive/My Drive/vgg16_best_params.pth')

        print(f"최고 정확도 갱신: {best_accuracy:.2f}% (vgg16_best_params.pth 파일 저장)")

total_end_time = time.time()

print(f"최종 최고 정확도: {best_accuracy:.2f}%")

사전 학습된 'features' 블록을 동결(freeze)합니다.
'classifier' 블록의 파라미터만 학습을 위해 옵티마이저에 전달합니다.
--- 20 Epoch Training Start (Classifier만 학습) ---
--- Epoch 1/20 완료 (소요 시간: 32.69초) ---
Training Loss: 1.0273 | Training Accuracy: 65.69 %
★ Validation Accuracy: 75.22% ★
최고 정확도 갱신: 75.22% (vgg16_best_params.pth 파일 저장)
--- Epoch 2/20 완료 (소요 시간: 35.92초) ---
Training Loss: 0.9066 | Training Accuracy: 70.46 %
★ Validation Accuracy: 76.50% ★
최고 정확도 갱신: 76.50% (vgg16_best_params.pth 파일 저장)
--- Epoch 3/20 완료 (소요 시간: 36.22초) ---
Training Loss: 0.8827 | Training Accuracy: 71.43 %
★ Validation Accuracy: 77.04% ★
최고 정확도 갱신: 77.04% (vgg16_best_params.pth 파일 저장)
--- Epoch 4/20 완료 (소요 시간: 34.68초) ---
Training Loss: 0.8726 | Training Accuracy: 72.05 %
★ Validation Accuracy: 77.10% ★
최고 정확도 갱신: 77.10% (vgg16_best_params.pth 파일 저장)
--- Epoch 5/20 완료 (소요 시간: 34.19초) ---
Training Loss: 0.8453 | Training Accuracy: 72.92 %
★ Validation Accuracy: 77.64% ★
최고 정확도 갱신: 77.64% (vgg16_best_params.pth 파일 저장)
--- Epoch 6/20 완료 (소요 시간: